## Lab2 exercise: evaluator-optimizer pattern 

Today we will work with lots of models! This is a way to get comfortable with APIs.

In [ ]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# Always remember to do this!
load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}") 
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
question = response.choices[0].message.content
display(Markdown(question))

## Calling LLMs from multple providers

We are about to call LLMs from many other providers.
They all provide API endpoints that are compatible with OpenAI, as explained in Guide 9 in the guides folder.
So we can simply use these endpoints as if we are using OpenAI.

Please note:

I'm going to use lots of LLMs from different providers, but you don't need to! This is only to show their abilities.

In [ ]:
# OpenAI Compatible URLs

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROK_BASE_URL = "https://api.x.ai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# OpenAI client libraries with the right base_url and key
# If this surprises you, please see Guide 9 in the Guides folder!

anthropic = OpenAI(api_key=anthropic_api_key, base_url=ANTHROPIC_BASE_URL)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=DEEPSEEK_BASE_URL)
gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)
groq = OpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)
grok = OpenAI(api_key=grok_api_key, base_url=GROK_BASE_URL)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [ ]:
# The API we know well
# Reasoning effort can be none, low, medium, high, or xhigh

model_name = "gpt-5.4-nano"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "claude-sonnet-4-6"

response = anthropic.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "gemini-3.1-flash-lite"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

In [ ]:
!ollama pull llama3.2

In [ ]:
import requests
requests.get('http://localhost:11434').content

In [ ]:
import requests
models = requests.get('http://localhost:11434/v1/models').json()
for model in models.get("data"):
    print(model.get("id"))

In [ ]:
from openai import OpenAI

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
model_name = "llama3.2:latest"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "gpt-oss:latest"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# So where are we?

print(len(competitors))
print(competitors)
print(answers)


In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
# Rebuild the judge prompt with the new answer
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
You must include all {len(competitors)} competitors in your ranking.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

judge_messages = [{"role": "user", "content": judge}]

# Judge and rank again
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=judge_messages)
results = response.choices[0].message.content
print(results)

In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

## And now for Grok!

Branded as "The most truth-seeking large language model in the world".. so let's use it as our LLM as a judge

In [ ]:
# Judgement time!
# Grok is "The most truth-seeking large language model in the world."

model_name = "grok-4.3"

response = grok.chat.completions.create(model=model_name, messages=judge_messages)
results = response.choices[0].message.content
print(results)


In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

## Exercise: Agentic design patterns

### Patterns already used in this lab
- **Prompt chaining:** one LLM writes the question, several models answer it, then a judge ranks the answers. Each step's output feeds the next.
- **Parallelization:** the same question goes to several models separately, and their answers are compared.
- **LLM-as-judge (evaluator):** a separate model ranks the answers and returns JSON.

### Pattern I added: Evaluator-optimizer
- **Generator:** `llama3.2` (local, via Ollama) answers the question.
- **Evaluator:** `gpt-5.4-mini` decides pass or fail and gives one sentence of feedback.
- **Loop:** if it fails, the generator revises its answer using the feedback (max 3 rounds).
- The refined answer is then entered into the original competition to see if it ranks higher.

**Result:** the evaluator didn't pass llama3.2's answer within 3 rounds, but the refined answer still ranked **6th of 7**, above the original llama3.2 (**7th**). Feedback improved a small local model, but didn't close the gap with larger models.

In [ ]:
# Generator: a local model answers, or revises using the judge's feedback
def generate(feedback=None, previous=None):
     msgs = [{"role": "user", "content": question}]
     if feedback:
          msgs += [
               {"role": "assistant", "content": previous}, 
               {"role": "user", "content": f"A judge gave this feedback: {feedback}\nRevise your answer. Keep it short."}
               ]
     response = ollama.chat.completions.create(model="llama3.2:latest", messages=msgs)
     return response.choices[0].message.content
          

In [ ]:
# Evaluator: a stronger model checks the answer and gives feedback
def evaluate(answer):
    prompt = f"""Question: {question}

Answer: {answer}

Is this answer insightful, correct and succinct? Respond with JSON only:
{{"pass": true or false, "feedback": "one sentence on how to improve"}}"""
    response = openai.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
# The loop: generate, evaluate, revise (at most 3 rounds)
answer = generate()
for i in range(3):
    verdict = evaluate(answer)
    display(Markdown(f"### Round {i+1}: pass = {verdict['pass']}\n**Feedback:** {verdict['feedback']}\n\n{answer}"))
    if verdict["pass"]:
        break
    answer = generate(verdict["feedback"], answer)

In [ ]:
# Record the refined answer
record("llama3.2 (refined)", answer)
print(competitors)

In [ ]:
def run_judge():
    # Combine every answer into one block of text
    together = ""
    for index, answer_text in enumerate(answers):
        together += f"# Response from competitor {index+1}\n\n{answer_text}\n\n"

    judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
You must include all {len(competitors)} competitors in your ranking.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else."""

    response = openai.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": judge}],
        response_format={"type": "json_object"},
    )
    ranks = json.loads(response.choices[0].message.content)["results"]
    for index, result in enumerate(ranks):
        print(f"Rank {index+1}: {competitors[int(result)-1]}")

In [ ]:
run_judge()